In [1]:
# A name is a pointer; rebinding one name doesn't touch another
a = 5
b = a   # b points to same object as a
a = 99  # a re-points to new object. b is untouched
print("a:", a, "b:", b)

a: 99 b: 5


In [5]:
# Aliasing a mutable object: one object, two names
list1 = [1, 2, 3]
list2 = list1       # NOT a copy - list2 points at the SAME list
list2.append(4)     # mutate the object in place
print("list1:", list1)
print("list2:", list2)
print("same object:", list1 is list2)


list1: [1, 2, 3, 4]
list2: [1, 2, 3, 4]
same object: True


In [8]:
#  Function arguments: mutation vs. rebinding
def add_item(seq):
    seq.append("new")       # MUTATES the passed-in object
data = ["a", "b"]
add_item(data)
print("after mutate:", data)

def rebind(seq):
    seq = ["totally", "new"]    # REBINDS the local name only
data2 = ["x", "y"]
rebind(data2)
print("after rebind:", data2)

after mutate: ['a', 'b', 'new']
['totally', 'new']
after rebind: ['x', 'y']


In [9]:
# Copy vs. deepcopy
import copy
original = [[1, 2], [3, 4]]     # a list containing two inner lists
shallow = copy.copy(original)   # new outer list, SAME inner lists
deep = copy.deepcopy(original)  # new outer list, NEW inner lists
original[0].append(999)         # mutate an inner list
print("original:", original)
print("shallow:", shallow)
print("deep:    ", deep)
# A deep copy recursively duplicates everything, so it's fully independent

original: [[1, 2, 999], [3, 4]]
shallow: [[1, 2, 999], [3, 4]]
deep:     [[1, 2], [3, 4]]


### Edge Cases

In [11]:
# 1 — The mutable default argument (the most famous Python trap)
def add_to(item, target=[]):
    target.append(item)
    return target
print(add_to(1))
print(add_to(2))
print(add_to(3))
# the default value [] is created once, 
# when the function is defined — not each time it's called. 

[1]
[1, 2]
[1, 2, 3]


In [12]:
# 2 — Small-integer caching, and why the demo is subtle
a = 256
b = 256
print("256 is 256:", a is b)    # True

256 is 256: True


In [13]:
c = 257
d = int(257)    # forces a fresh object, defeats literal-folding
print("257 is 257:", c is d)    # False

257 is 257: False


In [14]:
print(c == d)
# The takeaway isn't the cache boundary trivia — it's the rule it teaches: 
# never use is to compare values. Use ==. is is for identity only (and for None).

True


In [16]:
# String interning(caching)
s1 = "hello"
s2 = "hello"
print(s1 is s2)     # True

True


In [18]:
# 4 — A tuple containing a list (the "immutable isn't always frozen" case)
t = (1, 2, [3, 4])      # allowed - mutating the inner list
print("after:", t)
# t[0] = 99               # not allowed - rebinding a tuple slot
# 'tuple' object does not support item assignment

after: (1, 2, [3, 4])


In [23]:
t[2].append(5)

In [24]:
t

(1, 2, [3, 4, 5])

<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8">

  <p style="margin:0 0 16px 0; color:#cba6f7; font-weight:bold; font-size:1.05em;">📦 The Box and Arrow Analogy</p>
  <p style="margin:0 0 12px 0;">Think of a tuple as a <strong style="color:#89b4fa">locked storage unit with fixed slots.</strong></p>
  <ul style="margin:0 0 14px 0; padding-left:20px; line-height:2.2">
    <li>Slot 0 → solid gold brick (integer <code>1</code>)</li>
    <li>Slot 1 → solid gold brick (integer <code>2</code>)</li>
    <li>Slot 2 → cardboard box (the list <code>[3, 4]</code>)</li>
  </ul>
  <ul style="margin:0 0 18px 0; padding-left:20px; line-height:2.2">
    <li><strong style="color:#f38ba8">Why <code>t[0] = 99</code> fails</strong> — you're trying to rip out the gold brick and replace it. The locked slot says no.</li>
    <li><strong style="color:#a6e3a1">Why <code>t[2].append(5)</code> works</strong> — Slot 2 still holds the exact same cardboard box. You reached inside the box and tossed a number in. The tuple's slot never changed.</li>
  </ul>

  <p style="margin:0 0 12px 0; color:#cba6f7; font-weight:bold;">Why This Matters — Hashability &amp; Dictionary Keys</p>
  <p style="margin:0 0 14px 0;">To use something as a dict key or set member, it must be <strong style="color:#89b4fa">hashable</strong> — its value can never change so Python can assign it a permanent ID. A tuple containing a mutable list breaks this guarantee.</p>

  <p style="margin:0 0 8px 0; color:#f38ba8; font-weight:bold;">❌ The Broken Dictionary Key:</p>
  <div style="background:#181825; border-radius:6px; padding:14px 16px; margin:0 0 14px 0; border-left:4px solid #f38ba8;">
    <pre style="margin:0; color:#cdd6f4; line-height:2; overflow-x:auto">room_info = (<span style="color:#fab387">101</span>, [<span style="color:#a6e3a1">"Alice"</span>, <span style="color:#a6e3a1">"Bob"</span>])
hotel_registry = {}

<span style="color:#cba6f7">try</span>:
    hotel_registry[room_info] = <span style="color:#a6e3a1">"Occupied"</span>
<span style="color:#cba6f7">except</span> TypeError <span style="color:#cba6f7">as</span> e:
    <span style="color:#89b4fa">print</span>(f<span style="color:#a6e3a1">"Error: {e}"</span>)

<span style="color:#6c7086"># Error: unhashable type: 'list'</span></pre>
  </div>
  <p style="margin:0 0 16px 0;">Python stops you immediately — if it allowed this and you later changed the guest list, the dictionary would get completely lost trying to find that key again because its identity shifted.</p>

  <p style="margin:0 0 8px 0; color:#a6e3a1; font-weight:bold;">✅ The Fix — Deep Immutability:</p>
  <div style="background:#181825; border-radius:6px; padding:14px 16px; margin:0 0 14px 0; border-left:4px solid #a6e3a1;">
    <pre style="margin:0; color:#cdd6f4; line-height:2; overflow-x:auto"><span style="color:#6c7086"># Tuples all the way down</span>
perfectly_safe_tuple = (<span style="color:#fab387">1</span>, <span style="color:#fab387">2</span>, (<span style="color:#fab387">3</span>, <span style="color:#fab387">4</span>))

hotel_registry = {perfectly_safe_tuple: <span style="color:#a6e3a1">"Occupied"</span>}
<span style="color:#89b4fa">print</span>(<span style="color:#a6e3a1">"Success:"</span>, hotel_registry)</pre>
  </div>

  <p style="margin:0; border-left:4px solid #f38ba8; padding-left:12px">
    <strong style="color:#f38ba8">The rule:</strong> a tuple only protects its immediate layer. If you put something mutable inside it, that inner object can still change — and your hashability guarantee is gone.
  </p>

</div>

In [32]:
# This tuple contains a mutable list
room_info = (101, ["Alice", "Bob"])

# Let's try to use it as a dictionary key to store room status
hotel_registry = {}

try:
    hotel_registry[room_info] = "Occupied"
except TypeError as e:
    print(f"Error: {e}")

print("Memory Address (id):", id(room_info))
# print("Hash Value 1st time:", hash(room_info))
# print("Hash Value 2nd time:", hash(room_info))

Error: unhashable type: 'list'
Memory Address (id): 2100919112064


In [33]:
room_info[1].append("Charlie")

In [34]:
room_info

(101, ['Alice', 'Bob', 'Charlie'])

In [35]:
print("Memory Address (id):", id(room_info))

Memory Address (id): 2100919112064


In [38]:
class SneakyKey:
    def __init__(self, name):
        self.name = name

    # We tell Python to use the name string to calculate the hash
    def __hash__(self):
        return hash(self.name)

    # Required for dictionary lookups
    def __eq__(self, other):
        return self.name == other.name

# 1. Create our key
my_key = SneakyKey("Alice")

# 2. Store it in a dictionary
hotel_registry = {my_key: "Room 101"}
print("1. Memory Address of key:", id(my_key))
print("2. Stored successfully. Value is:", hotel_registry[my_key])

# 3. Sneakily change the internal data of the key!
my_key.name = "Charlie" 

print("3. Memory Address is STILL the same:", id(my_key))

# 4. Try to find it again
try:
    print("4. Looking up the key...")
    print(hotel_registry[my_key])
except KeyError:
    print("❌ ERROR: KeyError! Python got totally lost and couldn't find the key!")

# 5. Prove it's still technically in there
print("5. Dictionary contents:", hotel_registry)

1. Memory Address of key: 2100918671344
2. Stored successfully. Value is: Room 101
3. Memory Address is STILL the same: 2100918671344
4. Looking up the key...
❌ ERROR: KeyError! Python got totally lost and couldn't find the key!
5. Dictionary contents: {<__main__.SneakyKey object at 0x000001E92882D7F0>: 'Room 101'}


<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8">

  <p style="margin:0 0 16px 0; color:#cba6f7; font-weight:bold; font-size:1.05em;">📦 The Box and Arrow Analogy</p>
  <p style="margin:0 0 12px 0;">Think of a tuple as a <strong style="color:#89b4fa">locked storage unit with fixed slots.</strong></p>
  <ul style="margin:0 0 14px 0; padding-left:20px; line-height:2.2">
    <li>Slot 0 → solid gold brick (integer <code>1</code>)</li>
    <li>Slot 1 → solid gold brick (integer <code>2</code>)</li>
    <li>Slot 2 → cardboard box (the list <code>[3, 4]</code>)</li>
  </ul>
  <ul style="margin:0 0 18px 0; padding-left:20px; line-height:2.2">
    <li><strong style="color:#f38ba8">Why <code>t[0] = 99</code> fails</strong> — you're trying to rip out the gold brick and replace it. The locked slot says no.</li>
    <li><strong style="color:#a6e3a1">Why <code>t[2].append(5)</code> works</strong> — Slot 2 still holds the exact same cardboard box. You reached inside the box and tossed a number in. The tuple's slot never changed.</li>
  </ul>

  <p style="margin:0 0 12px 0; color:#cba6f7; font-weight:bold;">Why This Matters — Hashability &amp; Dictionary Keys</p>
  <p style="margin:0 0 14px 0;">To use something as a dict key or set member, it must be <strong style="color:#89b4fa">hashable</strong> — its value can never change so Python can assign it a permanent ID. A tuple containing a mutable list breaks this guarantee.</p>

  <p style="margin:0 0 8px 0; color:#f38ba8; font-weight:bold;">❌ The Broken Dictionary Key:</p>
  <div style="background:#181825; border-radius:6px; padding:14px 16px; margin:0 0 14px 0; border-left:4px solid #f38ba8;">
    <pre style="margin:0; color:#cdd6f4; line-height:2; overflow-x:auto">room_info = (<span style="color:#fab387">101</span>, [<span style="color:#a6e3a1">"Alice"</span>, <span style="color:#a6e3a1">"Bob"</span>])
hotel_registry = {}

<span style="color:#cba6f7">try</span>:
    hotel_registry[room_info] = <span style="color:#a6e3a1">"Occupied"</span>
<span style="color:#cba6f7">except</span> TypeError <span style="color:#cba6f7">as</span> e:
    <span style="color:#89b4fa">print</span>(f<span style="color:#a6e3a1">"Error: {e}"</span>)

<span style="color:#6c7086"># Error: unhashable type: 'list'</span></pre>
  </div>
  <p style="margin:0 0 16px 0;">Python stops you immediately — if it allowed this and you later changed the guest list, the dictionary would get completely lost trying to find that key again because its identity shifted.</p>

  <p style="margin:0 0 8px 0; color:#a6e3a1; font-weight:bold;">✅ The Fix — Deep Immutability:</p>
  <div style="background:#181825; border-radius:6px; padding:14px 16px; margin:0 0 14px 0; border-left:4px solid #a6e3a1;">
    <pre style="margin:0; color:#cdd6f4; line-height:2; overflow-x:auto"><span style="color:#6c7086"># Tuples all the way down</span>
perfectly_safe_tuple = (<span style="color:#fab387">1</span>, <span style="color:#fab387">2</span>, (<span style="color:#fab387">3</span>, <span style="color:#fab387">4</span>))

hotel_registry = {perfectly_safe_tuple: <span style="color:#a6e3a1">"Occupied"</span>}
<span style="color:#89b4fa">print</span>(<span style="color:#a6e3a1">"Success:"</span>, hotel_registry)</pre>
  </div>

  <p style="margin:0; border-left:4px solid #f38ba8; padding-left:12px">
    <strong style="color:#f38ba8">The rule:</strong> a tuple only protects its immediate layer. If you put something mutable inside it, that inner object can still change — and your hashability guarantee is gone.
  </p>

</div>

In [39]:
#  5 — == vs is, side by side
l1 = [1, 2, 3]
l2 = [1, 2, 3]
print("l1 == l2:", l1 == l2)    # equal VALUES
print("l1 is l2:", l1 is l2)    # different OBJECTS

l1 == l2: True
l1 is l2: False


`==` asks "same value?" `is` asks "same object in memory?" 

Two separately built lists are equal but not identical. 

In [42]:
# 6 — Augmented assignment (+=) behaves differently on list vs. tuple
lst = [1, 2]
print("list id before:", id(lst))
lst += [3]                      # mutates IN PLACE - same object
print("lst id after: ", id(lst))
print("lst: ", lst)

tup = (1, 2)
print("tuple id before:", id(tup))
tup += (3,)                     # creates a NEW object - rebinds
print("tuple id after: ", id(tup))
print("tup: ", tup)

list id before: 2100919609088
lst id after:  2100919609088
lst:  [1, 2, 3]
tuple id before: 2100918272320
tuple id after:  2100918303168
tup:  (1, 2, 3)


In [43]:
t = (1, 2, [3, 4])
try:
    t[2] += [5]
except Exception as e:
    print(e)

'tuple' object does not support item assignment


In [44]:
print(t)

(1, 2, [3, 4, 5])


### Common Traps

In [45]:
# Trap 1 — Silent argument mutation corrupts your source data.
def normalize(row):
    row.append("processed")
    return row
master = ["id1", "value"]
result = normalize(master)
print(master)       # ['id1', 'value', 'processed']

['id1', 'value', 'processed']


In [67]:
# Trap 2 — Aliasing vs. copying.
a = [1, 2, 3, 4]
b = a       # alias: b tracks very change to a
c = a[:]    # copy: independent
print("b:", b, "id: " ,id(b))
print("c:",c, "id: ", id(c))
# c = c.append(5) #.append() method modifies the list in-place and returns None. 
                # By assigning c.append(5) back to c,
                # you are overwriting your beautiful list with that None return value.
c.append(5)
print(c)

b: [1, 2, 3, 4] id:  2100917700160
c: [1, 2, 3, 4] id:  2100917712128
[1, 2, 3, 4, 5]


In [68]:
# Trap 3 — Mutable class attribute shared across instances.
class Cart:
    items = []      # SHARED by every Cart
    def add(self, x): self.items.append(x)
c1, c2 = Cart(), Cart()
c1.add("apple")
print(c2.items)    # ['apple']  <- c2 has c1's data

['apple']


In [69]:
# Trap 4 — The [[0]*3]*3 grid trap.
grid = [[0]*3 for _ in range(3)]
print(grid)

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]


In [70]:
bad = [[0]*3]*3
print(bad)

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]


In [71]:
grid[0][0] = 1
print(grid)

[[1, 0, 0], [0, 0, 0], [0, 0, 0]]


In [72]:
bad[0][0] = 1
print(bad)

[[1, 0, 0], [1, 0, 0], [1, 0, 0]]


In [79]:
# Trap 5 — Late binding closures in a loop.
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])

[2, 2, 2]


In [80]:
funcs = [lambda i = i: i for i in range(3)]
print([f() for f in funcs])

[0, 1, 2]


### EXERCISE

In [81]:
# 1. Predict the output
p = [1, 2, 3]
q = p           # here copies the pointer, not the list.
q.append(4)
print(q)       # copies the list not pointer

[1, 2, 3, 4]


<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8">
  <p style="margin:0 0 12px 0;">
    <strong style="color:#f38ba8">Correction:</strong> <code>q = p</code> copies the <strong style="color:#f38ba8">pointer</strong>, not the list. There is one list object — <code>p</code> and <code>q</code> are two name tags on the same object.
  </p>
  <p style="margin:0; border-left:4px solid #a6e3a1; padding-left:12px">
    <strong style="color:#a6e3a1">The proof:</strong> <code>p</code> also shows the <code>4</code> after <code>q.append(4)</code>. If <code>q = p</code> had copied the list, <code>p</code> would still be <code>[1, 2, 3]</code>. The fact that it isn't proves both names point at the same object in memory.
  </p>
</div>

<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8">
  <p style="margin:0 0 12px 0;">
    <strong style="color:#f38ba8">Your comment was backwards — and that's the whole point of the problem.</strong>
  </p>
  <ul style="margin:0 0 14px 0; padding-left:20px; line-height:2.2">
    <li><code>q = p</code> copies the <strong style="color:#f38ba8">pointer</strong>, not the list</li>
    <li>There is <strong style="color:#89b4fa">one list object</strong> in memory — <code>p</code> and <code>q</code> are just two name tags stuck on the same object</li>
    <li>If it had copied the list, <code>p</code> would still be <code>[1, 2, 3]</code> — untouched</li>
    <li>The fact that <code>p</code> also shows <code>[1, 2, 3, 4]</code> after <code>q.append(4)</code> <strong style="color:#a6e3a1">proves it's the same object</strong></li>
  </ul>
  <div style="background:#181825; border-radius:6px; padding:14px 16px; border-left:4px solid #f38ba8;">
    <pre style="margin:0; color:#cdd6f4; line-height:2; overflow-x:auto">p = [<span style="color:#fab387">1</span>, <span style="color:#fab387">2</span>, <span style="color:#fab387">3</span>]
q = p              <span style="color:#6c7086"># copies the POINTER — both names point at the same list</span>
q.append(<span style="color:#fab387">4</span>)
<span style="color:#89b4fa">print</span>(p)           <span style="color:#6c7086"># [1, 2, 3, 4] — p sees it too, same object</span>
<span style="color:#89b4fa">print</span>(p <span style="color:#cba6f7">is</span> q)     <span style="color:#6c7086"># True — not two lists, one list, two names</span></pre>
  </div>
</div>

In [82]:
# 2. True or False, and why
x = (1, 2, 3)
print(x[0] is 1)

True


<>:3: SyntaxWarning: "is" with 'int' literal. Did you mean "=="?
<>:3: SyntaxWarning: "is" with 'int' literal. Did you mean "=="?
C:\Users\2321764\AppData\Local\Temp\ipykernel_21920\2084949947.py:3: SyntaxWarning: "is" with 'int' literal. Did you mean "=="?
  print(x[0] is 1)


In [83]:
y = "data"
z = "data"
print(y is z) #is asks same object in memory?

True


<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8">
  <p style="margin:0 0 12px 0;">
    <strong style="color:#a6e3a1">You got the identity-vs-memory idea right.</strong> The <code>SyntaxWarning</code> is Python itself enforcing the rule.
  </p>
  <ul style="margin:0 0 14px 0; padding-left:20px; line-height:2.2">
    <li><code>x[0] is 1</code> prints <code>True</code> — but only because of the <strong style="color:#89b4fa">small-int cache</strong> (integers −5 to 256 are pre-cached, so <code>1</code> always hits the same object)</li>
    <li><code>y is z</code> prints <code>True</code> — but only because of <strong style="color:#89b4fa">string interning</strong> (Python reuses objects for short, simple strings at compile time)</li>
  </ul>
  <p style="margin:0 0 12px 0;">
    Both are <strong style="color:#f38ba8">CPython implementation details</strong> — not language guarantees. A different Python version, a different runtime (PyPy, Jython), or a string built at runtime can silently break both.
  </p>
  <p style="margin:0; border-left:4px solid #cba6f7; padding-left:12px">
    <strong style="color:#cba6f7">The rule:</strong> <code>is</code> is for identity checks — <code>None</code>, <code>True</code>, <code>False</code>, sentinel objects. For everything else, use <code>==</code>. The <code>SyntaxWarning</code> exists precisely because beginners get lucky with literals and think they understand <code>is</code> — until they don't.
  </p>
</div>

In [89]:
# 3 (Medium) — Fix the bug. This function is supposed to return a new list with the item added, 
# leaving the input untouched. It doesn't. Fix it two different ways.
def with_item(lst, item):
    lst.append(item)
    return lst
lst = [1, 2, 3]
lst_copy = lst[:]
i = with_item(lst_copy, 4)
print(i)
print(lst)

[1, 2, 3, 4]
[1, 2, 3]


In [91]:
# 3 (Medium) — Fix the bug. This function is supposed to return a new list with the item added, 
# leaving the input untouched. It doesn't. Fix it two different ways.
def with_item(lst, item):
    lst.append(item)
    return lst
lst = [1, 2, 3]
lst_copy = lst.copy()
i = with_item(lst_copy, 4)
print(i)
print(lst)

[1, 2, 3, 4]
[1, 2, 3]


<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8">
  <p style="margin:0 0 12px 0;">
    <strong style="color:#a6e3a1">Your fix works — but you fixed it at the call site.</strong> The problem asked the function itself not to mutate. There's a meaningful difference:
  </p>
  <ul style="margin:0 0 14px 0; padding-left:20px; line-height:2.2">
    <li><strong style="color:#f38ba8">Call-site fix</strong> — every caller has to remember to copy before passing. One forgetful caller, one silent bug.</li>
    <li><strong style="color:#a6e3a1">Function-level fix</strong> — safety lives inside the function. No caller can ever get burned, regardless of how they call it.</li>
  </ul>

  <div style="background:#181825; border-radius:6px; padding:14px 16px; margin:0 0 14px 0; border-left:4px solid #a6e3a1;">
    <pre style="margin:0; color:#cdd6f4; line-height:2; overflow-x:auto"><span style="color:#6c7086"># Option 1 — copy inside the function</span>
<span style="color:#cba6f7">def</span> <span style="color:#89b4fa">with_item</span>(lst, item):
    new = lst.copy()      <span style="color:#6c7086"># lst[:] or list(lst) also work</span>
    new.append(item)
    <span style="color:#cba6f7">return</span> new

<span style="color:#6c7086"># Option 2 — no mutation at all (tightest)</span>
<span style="color:#cba6f7">def</span> <span style="color:#89b4fa">with_item</span>(lst, item):
    <span style="color:#cba6f7">return</span> lst + [item]   <span style="color:#6c7086"># + builds a brand-new list every time</span></pre>
  </div>

  <p style="margin:0; border-left:4px solid #cba6f7; padding-left:12px">
    <strong style="color:#cba6f7">Interview answer:</strong> "I make the function <strong style="color:#89b4fa">pure</strong> — it never mutates its input, it returns a new object. That way the guarantee is in the contract of the function, not in the discipline of every caller."
  </p>
</div>

In [103]:
# 4 (Medium) — Why are these different, and which do you want for a 3×3 matrix?
a = [[0]*3 for _ in range(3)]
# comprehension RUNS [0]*3 three times -> 3 distinct lists
b = [[0]*3]*3
# [0]*3 runs ONCE -> then *3 copies that ONE pointer 3x
a[0][0] = 1
b[0][0] = 1
print(a)
print(len(set(id(row) for row in a)))   # 3  -> three different objects
print(b)                                # 1  -> one object, aliased thrice
print(len(set(id(row) for row in b)))

[[1, 0, 0], [0, 0, 0], [0, 0, 0]]
3
[[1, 0, 0], [1, 0, 0], [1, 0, 0]]
1


Always build nested structures with a comprehension, never with *.

In [104]:
# Predict the full output and the final state.
def process(data, cache={}):
    key = len(data)
    if key not in cache:
        cache[key] = data
    data.append("x")
    return cache

print(process([1]))
print(process([1, 2]))
print(process([9]))

{1: [1, 'x']}
{1: [1, 'x'], 2: [1, 2, 'x']}
{1: [1, 'x'], 2: [1, 2, 'x']}


Hint: two separate object-model effects stack here — the mutable default cache persisting across calls, AND cache[key] = data storing a reference (not a copy) of a list you then mutate. Trace what cache holds vs. what the stored lists actually contain after each call. This one is genuinely tricky; take your time.

<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8">

  <p style="margin:0 0 12px 0;">
    <strong style="color:#f38ba8">Because <code>cache[key] = data</code> does not store the list. It stores a pointer to the list.</strong> This is Part 1's entire model, now biting at depth. Walk it:
  </p>

  <div style="background:#181825; border-radius:6px; padding:14px 16px; margin:0 0 14px 0; border-left:4px solid #f38ba8;">
    <pre style="margin:0; color:#cdd6f4; line-height:2; overflow-x:auto">cache[key] = data    <span style="color:#6c7086"># cache[1] now points at the SAME object 'data' points at</span>
data.append(<span style="color:#a6e3a1">"x"</span>)     <span style="color:#6c7086"># mutate that one shared object IN PLACE</span></pre>
  </div>

  <p style="margin:0 0 12px 0;">
    There is only one list. <code>cache[1]</code> and the parameter <code>data</code> are two name tags on it (just like <code>p</code> and <code>q</code> in Problem 1). When you append through one name, the other name sees it — because they're the same object. The dict didn't take a snapshot; it took a reference.
  </p>

  <p style="margin:0 0 8px 0; color:#cba6f7; font-weight:bold;">Trace for the first call <code>process([1])</code>:</p>
  <div style="background:#181825; border-radius:6px; padding:14px 16px; margin:0 0 14px 0; border-left:4px solid #89b4fa;">
    <pre style="margin:0; color:#cdd6f4; line-height:2; overflow-x:auto">key = <span style="color:#fab387">1</span>, not in cache  -> cache[<span style="color:#fab387">1</span>] = data   <span style="color:#6c7086">(cache = {1: [1]} ... but [1] IS the live list)</span>
data.append(<span style="color:#a6e3a1">"x"</span>)       -> the shared list becomes [<span style="color:#fab387">1</span>, <span style="color:#a6e3a1">"x"</span>]
                       -> so cache is now {<span style="color:#fab387">1</span>: [<span style="color:#fab387">1</span>, <span style="color:#a6e3a1">"x"</span>]}
return cache           -> {<span style="color:#fab387">1</span>: [<span style="color:#fab387">1</span>, <span style="color:#a6e3a1">'x'</span>]}</pre>
  </div>
  <p style="margin:0 0 14px 0;">That's your "where did the x come from" — it came from mutating the very object the dict is pointing at, after storing the pointer.</p>

  <p style="margin:0 0 8px 0; color:#cba6f7; font-weight:bold;">Now the part that makes this a senior question — the mutable default <code>cache={}</code> persists across calls (Trap 1 / Golden Rule 1). It's the same dict every call:</p>
  <div style="background:#181825; border-radius:6px; padding:14px 16px; margin:0 0 0 0; border-left:4px solid #cba6f7;">
    <pre style="margin:0; color:#cdd6f4; line-height:2; overflow-x:auto">process([<span style="color:#fab387">1</span>])      <span style="color:#6c7086"># cache={1:[1,'x']}</span>
process([<span style="color:#fab387">1</span>, <span style="color:#fab387">2</span>])   <span style="color:#6c7086"># key=2 new -> cache={1:[...], 2:[1,2,'x']}</span>
process([<span style="color:#fab387">9</span>])      <span style="color:#6c7086"># key=1 ALREADY in cache -> does NOT overwrite!</span>
                  <span style="color:#6c7086"># cache[1] stays pointing at the OLD list,</span>
                  <span style="color:#6c7086"># but data=[9] still gets "x" appended -> [9,'x'] (not stored)</span></pre>
  </div>

</div>

In [105]:
def add_features(row):
    row.append(row[0]*2)
    return row
raw = [3, 5]
features = add_features(raw)
print(features)

[3, 5, 6]


In [114]:
def add_features(row):
    print(row + [row[0]*2])
    print(raw)
raw = [3, 5]
features = add_features(raw)

[3, 5, 6]
[3, 5]


In [121]:
def train(params=None):
    # If no dictionary was passed, create a brand new for this run
    if params is None:
        params = {"lr": 0.01}
    
    params["lr"] *= 0.1
    return params

# Now each run is completely independent:
print(train())   # {'lr': 0.001}
print(train())   # {'lr': 0.001}  <- Fixed! No longer decaying across calls

{'lr': 0.001}
{'lr': 0.001}


By using params=None, the default value itself is immutable. The actual dictionary creation {"lr": 0.01} is moved inside the function body. This ensures that a brand-new dictionary is allocated in memory every single time train() is invoked without arguments.

In [126]:
# NumPy slicing returns a view that shares memory with the original:
import numpy as np
a = np.array([1, 2, 3, 4])
view = a[1:3]   # shares memory
view[0] = 99
print("view:",view)
print("a:", a)

view: [99  3]
a: [ 1 99  3  4]


When pandas warns you "you're setting on a copy of a slice," it's warning about exactly the names-vs-objects ambiguity from Part 1. 

### Code Challenges

In [ ]:
# 1. Swap two variables without a temp variable.
a,b = b,a   # tuple packing/unpacking; RHS builds a tuple first
# 1, 2 → 2, 1. Works because the right side is fully evaluated into a tuple 
# before any binding happens.

In [140]:
# 2. Remove duplicates from a list while repserving order.
def dedup(seq):
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

In [141]:
dedup([3,1,3,2,1,5])

[3, 1, 2, 5]

<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8">
  <p style="margin:0 0 12px 0;">
    <code>set</code> membership is <strong style="color:#a6e3a1">O(1)</strong>, so the full loop is <strong style="color:#a6e3a1">O(n)</strong>.
  </p>
  <p style="margin:0; border-left:4px solid #cba6f7; padding-left:12px">
    Directly relevant to your <strong style="color:#cba6f7">OpsRAG deduplication work</strong> — same pattern: <strong style="color:#89b4fa">seen set + ordered output</strong>.
  </p>
</div>

In [142]:
# One-liner version
print(list(dict.fromkeys([3,1,3,2,1,5])))

[3, 1, 2, 5]


<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8">

  <ol style="margin:0 0 14px 0; padding-left:22px; line-height:2.4">
    <li><strong style="color:#89b4fa"><code>dict.fromkeys()</code></strong> is a built-in method that creates a new dictionary using the items from an iterable as the dictionary keys. Since dictionaries don't require values for this method, it defaults all the values to <code>None</code>.</li>
    <li><strong style="color:#89b4fa">Keys must be unique:</strong> because dictionary keys cannot have duplicates, Python automatically discards any repeating elements (<code>3</code> and <code>1</code>) as it processes the list.</li>
    <li><strong style="color:#89b4fa">Order is preserved:</strong> because Python dicts preserve insertion order, the remaining keys stay in the exact sequence they first appeared: <code>{3: None, 1: None, 2: None, 5: None}</code>.</li>
  </ol>

  <div style="background:#181825; border-radius:6px; padding:14px 16px; margin:0 0 14px 0; border-left:4px solid #89b4fa;">
    <pre style="margin:0; color:#cdd6f4; line-height:2; overflow-x:auto"><span style="color:#89b4fa">list</span>(...)</pre>
  </div>

  <ol start="4" style="margin:0 0 14px 0; padding-left:22px; line-height:2.4">
    <li>Finally, <code>list()</code> grabs just the keys of that dictionary and converts them back into a clean list: <code>[3, 1, 2, 5]</code>.</li>
  </ol>

  <p style="margin:0 0 8px 0; color:#cba6f7; font-weight:bold;">Why use this over <code>list(set(...))</code>?</p>
  <p style="margin:0;">While <code>list(set(x))</code> is also a one-liner that removes duplicates, <code>set()</code> does not preserve order. A set will scramble your list based on hash values, whereas <code>dict.fromkeys()</code> strips the duplicates while keeping your original order perfectly intact.</p>

</div>

In [144]:
# 3. Write a function that returns a new list with an item appended, without mutating the input.
def append_pure(lst, item):
    return lst + [item]     # + builds a new list

print(append_pure([1,2], 3))

[1, 2, 3]
[1, 2, 3]


In [149]:
# To check original list was mutated
# The content Check
def append_pure(lst, item):
    return lst + [item]     # + builds a new list
lst = [1,2]
new_lst = append_pure(lst, 3)
print("Original list:",lst)
print("Returned list:",new_lst)

Original list: [1, 2]
Returned list: [1, 2, 3]


In [150]:
# Identity Check
# In Python, every object has a unique memory address accessible via id().
# Check if they point to the same place in memory
print(id(lst) == id(new_lst))

False


In [154]:
# What would a mutating version look like?
# If you had used .append(), it would modify the list in place and return None.
# DANGER: This mutates the original list!
def append_mutator(lst, item):
    lst.append(item)
    return lst
lst = [1, 2]
new_lst = append_mutator(lst, 3)
print('new_list:', new_lst)
print('original list:', lst)

new_list: [1, 2, 3]
original list: [1, 2, 3]


In [157]:
# Predict the output, then explain
def f(x, acc=[]):
    acc.append(x)
    return acc
print(f(1)); print(f(2)); print(f(3, []))

[1]
[1, 2]
[3]


<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8">
  <p style="margin:0 0 12px 0;">
    Calls 1 and 2 share the persistent default list → they accumulate. Call 3 passes its own fresh list, so it's <code>[3]</code> and doesn't touch the shared default.
  </p>
  <p style="margin:0; border-left:4px solid #cba6f7; padding-left:12px">
    Demonstrates the trap — and that explicitly passing an argument bypasses the default entirely.
  </p>
</div>

In [162]:
# Flatten one level of nesting: [[1,2],[3,4],[5]] -> [1,2,3,4,5].
def flatten(nested):
    return [x for sub in nested for x in sub]

lst = [[1, 2], [3, 4], [5]]
print(flatten(lst))

[1, 2, 3, 4, 5]


In [170]:
# You have m = [[1,2],[3,4]]. Make s a shallow copy and d a deep copy, 
# then mutate m[0][0] = 99. What does each show?
import copy
m = [[1, 2], [3, 4]]
s = copy.copy(m)
d = copy.deepcopy(m)
m[0][0] = 99
print(s, '\n', d)

[[99, 2], [3, 4]] 
 [[1, 2], [3, 4]]


<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8">
  <p style="margin:0;">
    The inner list is <strong style="color:#f38ba8">shared</strong> by the shallow copy — so mutating it shows through.
  </p>
</div>

In [173]:
# Why does d[(1,2,3)] = "ok" work but d[(1,[2])] = "no" fail?
d ={}
d[(1,2,3)] = "ok"
print(d)
try:
    d[(1, [2])] = "no"
except Exception as e:
    print(e)

{(1, 2, 3): 'ok'}
unhashable type: 'list'


<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8">
  <p style="margin:0 0 12px 0;">
    Output: <code>unhashable type: 'list'</code>. A tuple is hashable only if <strong style="color:#f38ba8">all its contents are hashable</strong>. The inner list makes the whole key unhashable.
  </p>
  <p style="margin:0; border-left:4px solid #cba6f7; padding-left:12px">
    Ties Q5 + Q6 together.
  </p>
</div>

In [175]:
# Write safe_update(d, key, value) that returns a new dict with the key set, 
# leaving the input dict unchanged — and prove the original is untouched.
def safe_update(d, key, value):
    new = d.copy()      # shallow copy of the dict
    new[key] = value
    return new

orig = {"a": 1}
result = safe_update(orig, "b", 2)
print('original list:',orig)
print('new list:',result)

original list: {'a': 1}
new list: {'a': 1, 'b': 2}


<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8">
  <p style="margin:0 0 12px 0;">
    <code>d.copy()</code> is <strong style="color:#f38ba8">shallow</strong> — if values are mutable and you mutate them, the original still sees it. For full isolation use <code>copy.deepcopy</code>.
  </p>
  <p style="margin:0 0 12px 0; border-left:4px solid #cba6f7; padding-left:12px">
    Knowing when shallow is enough vs. not is the <strong style="color:#cba6f7">senior signal</strong>.
  </p>
  <p style="margin:0; border-left:4px solid #a6e3a1; padding-left:12px">
    <strong style="color:#a6e3a1">Python 3.9+ one-liner:</strong> <code>{**d, key: value}</code>
  </p>
</div>

In [178]:
def safe_update(d, key, value):
    return {**d, key:value}
orig = {"a": 1}
result = safe_update(orig, "b", 2)

print("result:", result)  
print("orig:", orig)      

result: {'a': 1, 'b': 2}
orig: {'a': 1}


The outer braces {} allocates a brand new empty dictionary in memory.

The very first thing Python does is see the outer { }. This tells it to allocate a brand-new, empty dictionary in memory. Because it creates a new dictionary from scratch, your original dictionary d is left completely safe and untouched.

In [179]:
def safe_update(d, key, value):
    return d | {key: value}

# Testing it out:
orig = {"a": 1}
result = safe_update(orig, "b", 2)

print("result:", result)  # {'a': 1, 'b': 2}
print("orig:", orig)      # {'a': 1} <- Still safe and untouched!

result: {'a': 1, 'b': 2}
orig: {'a': 1}
